# EvidenceDoc: GPU training smoke test
Select a CUDA GPU runtime. Upload the provided EvidenceDoc.zip. This notebook has not been GPU-executed in the supplied build. Generated data only exercise the pipeline; replace them with reviewed real training/validation labels before claiming model improvement.
Begin with two steps; inspect memory and loss before a full run. GPU availability and capacity depend on your environment.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
destination = Path('/content/evidencedoc-project').resolve()
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for member in z.namelist():
        if not (destination / member).resolve().is_relative_to(destination):
            raise ValueError('Unsafe archive path')
    z.extractall(destination)
os.chdir(destination / 'EvidenceDoc')
print(Path.cwd())


In [ ]:
!nvidia-smi
%pip install -e '.[dev,train]'


In [ ]:
!python -m scripts.generate_data --count 60
!python -m scripts.train --dry-run


In [ ]:
!python -m scripts.train --config configs/qlora.yaml --max-steps 2


## After a successful smoke run
Inspect `run_metadata.json`, token masks, and loss. Then adjust the manifest in `configs/qlora.yaml` to your reviewed real dataset and run a complete training job. Evaluate the original and adapted checkpoints against the same locked real test set.

In [ ]:
# Uncomment only after checking the smoke run and dataset.
# !python -m scripts.train --config configs/qlora.yaml


In [ ]:
# Download the adapter and run metadata after a successful job.
import shutil
checkpoint = Path('checkpoints/evidencedoc-qwen3vl-2b')
if checkpoint.exists():
    archive = shutil.make_archive('/content/evidencedoc-adapter', 'zip', checkpoint)
    files.download(archive)
